# Day 044 Solution — Databases in Python

setup_engine, add_item, get_items, update_price, delete_item. All in-memory SQLite via SQLAlchemy 2.x ORM. Self-contained.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from sqlalchemy import create_engine, String, Float, Integer, select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session
from sqlalchemy.pool import StaticPool


class Base(DeclarativeBase):
    pass


class Item(Base):
    __tablename__ = 'items'
    id:       Mapped[int]   = mapped_column(primary_key=True)
    name:     Mapped[str]   = mapped_column(String(100))
    category: Mapped[str]   = mapped_column(String(50))
    price:    Mapped[float] = mapped_column()
    quantity: Mapped[int]   = mapped_column(default=0)

    def __repr__(self):
        return f'Item(id={self.id}, name={self.name!r}, price={self.price})'


def setup_engine(url='sqlite:///:memory:'):
    engine = create_engine(
        url,
        connect_args={'check_same_thread': False},
        poolclass=StaticPool,
    )
    Base.metadata.create_all(engine)
    return engine


def add_item(session, name, category, price, quantity=0):
    item = Item(name=name, category=category, price=price, quantity=quantity)
    session.add(item)
    session.commit()
    session.refresh(item)
    return item


def get_items(session, category=None):
    stmt = select(Item)
    if category is not None:
        stmt = stmt.where(Item.category == category)
    return list(session.execute(stmt).scalars().all())


def update_price(session, item_id, new_price):
    item = session.get(Item, item_id)
    if item is None:
        return None
    item.price = new_price
    session.commit()
    session.refresh(item)
    return item


def delete_item(session, item_id):
    item = session.get(Item, item_id)
    if item is None:
        return False
    session.delete(item)
    session.commit()
    return True

## Step 1 — Engine and Session

In [ ]:
engine  = setup_engine()
session = Session(engine)

from sqlalchemy import inspect as sa_inspect
tables = sa_inspect(engine).get_table_names()
assert 'items' in tables
print(f'Tables: {tables}')

## Step 2 — add_item

In [ ]:
laptop  = add_item(session, 'Laptop',     'Electronics', 999.99, 5)
phones  = add_item(session, 'Headphones', 'Electronics', 149.99, 12)
chair   = add_item(session, 'Desk Chair', 'Furniture',   349.00, 3)
book    = add_item(session, 'Bookcase',   'Furniture',   199.00, 8)
pens    = add_item(session, 'Pen Set',    'Stationery',   12.99, 50)

assert laptop.id is not None
assert laptop.name == 'Laptop'
assert abs(laptop.price - 999.99) < 0.01
print(f'Added: {laptop}')

## Step 3 — get_items

In [ ]:
all_items = get_items(session)
assert len(all_items) == 5
assert all(isinstance(i, Item) for i in all_items)
print(f'All items: {len(all_items)}')

elec = get_items(session, category='Electronics')
assert len(elec) == 2
assert all(i.category == 'Electronics' for i in elec)
print(f'Electronics: {[i.name for i in elec]}')

empty = get_items(session, category='NonExistent')
assert empty == []
print(f'NonExistent: {empty}')

## Step 4 — update_price

In [ ]:
updated = update_price(session, laptop.id, 849.99)
assert updated is not None
assert abs(updated.price - 849.99) < 0.01
print(f'Updated: {updated}')

not_found = update_price(session, 99999, 1.0)
assert not_found is None
print(f'Missing id: {not_found}')

## Step 5 — delete_item

In [ ]:
before = len(get_items(session))
deleted = delete_item(session, pens.id)
assert deleted is True
assert session.get(Item, pens.id) is None
after = len(get_items(session))
assert after == before - 1
print(f'Deleted pens. Before: {before}, after: {after}')

not_deleted = delete_item(session, 99999)
assert not_deleted is False
print(f'Missing id: {not_deleted}')

print('\nAll solution checks passed.')
session.close()